This notebook is to prove that data leakage does not occur throughout our pre-processing to training steps.

For this example, we will use the logistic regression pre-processing to training
#### Logistic Regression 


We will run the notebook parts prior to testing below

In [1]:
import numpy as np
import pandas as pd


import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from scipy.stats import ttest_rel
%matplotlib inline


plt.rcParams['figure.figsize'] = (6.0, 6.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

plt.style.use('ggplot')

# autoreload external python modules;
# see http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

try:
    lfs_data = pd.read_csv("src/data/LFS PUF April 2016.CSV")
except FileNotFoundError:
    print("Error: CSV file not found. Please make sure the file exists in the correct directory or provide the correct path.")
    exit()


## Preprocessing
has_null = lfs_data.apply(lambda col: col.str.isspace().sum() if col.dtype == 'object' else 0)

lfs_data.replace(r"^\s+$", -1, regex=True, inplace=True)
nan_counts_per_column = lfs_data.isna().sum()

int_convertible_columns = []

for col in lfs_data.columns:
    if lfs_data[col].dtypes == 'object':  
        try:
            float_vals = lfs_data[col].dropna().astype(float)
            if (float_vals % 1 == 0).all():
                int_convertible_columns.append(col)
        except ValueError:
            pass 

columns_to_convert = [
    'PUFC06_MSTAT', 'PUFC08_CURSCH', 'PUFC09_GRADTECH', 'PUFC10_CONWR', 'PUFC11_WORK', 
    'PUFC12_JOB', 'PUFC14_PROCC', 'PUFC16_PKB', 'PUFC17_NATEM', 'PUFC18_PNWHRS', 
    'PUFC19_PHOURS', 'PUFC20_PWMORE', 'PUFC21_PLADDW', 'PUFC22_PFWRK', 'PUFC23_PCLASS', 
    'PUFC24_PBASIS', 'PUFC25_PBASIC', 'PUFC26_OJOB', 'PUFC27_NJOBS', 'PUFC28_THOURS', 
    'PUFC29_WWM48H', 'PUFC30_LOOKW', 'PUFC31_FLWRK', 'PUFC32_JOBSM', 'PUFC33_WEEKS', 
    'PUFC34_WYNOT', 'PUFC35_LTLOOKW', 'PUFC36_AVAIL', 'PUFC37_WILLING', 'PUFC38_PREVJOB', 
    'PUFC40_POCC', 'PUFC41_WQTR', 'PUFC43_QKB', 'PUFNEWEMPSTAT'
]

for col in columns_to_convert:
    lfs_data[col] = lfs_data[col].astype(int) 

lfs_data.apply(lambda x: x.nunique())

lfs_data['PUFC07_GRADE'] = lfs_data['PUFC07_GRADE']
valid_codes = [
    0, 10,  # No Grade, Preschool
    210, 220, 230, 240, 250, 260, 280,  # Elementary
    310, 320, 330, 340, 350,  # High School
    410, 420,  # Post Secondary; If Graduate Specify
    810, 820, 830, 840,  # College; If Graduate Specify
    900,  # Post Baccalaureate
    np.nan
]
invalid_rows = lfs_data[~(lfs_data['PUFC07_GRADE'].isin(valid_codes))]

unique_invalid_values = invalid_rows['PUFC07_GRADE'].unique()

lfs_data.loc[~lfs_data['PUFC07_GRADE'].isin(valid_codes), 'PUFC07_GRADE'] = 700
print(lfs_data['PUFC07_GRADE'].unique())


[700]


We will be splitting our dataset into different samples, in which we will be training 5 folds and leaving 1 fold for testing at the end. Note that the # of splits in the data preparation is 6 as compared to our 5 in the notebook.

In [2]:
from src.preprocessing import prepare_data_kfold

target_col = 'PUFC11_WORK'
feature_cols = [
    'PUFC05_AGE', 'PUFC06_MSTAT', 'PUFC04_SEX', 
    'PUFC07_GRADE', 'PUFC08_CURSCH', 
    'PUFC38_PREVJOB', 'PUFC31_FLWRK',
    'PUFC30_LOOKW', 'PUFC34_WYNOT'
]

categorical_cols = feature_cols
n_splits = 6 
missing_value =-1
seed = 45

folds_data = prepare_data_kfold(lfs_data, target_col = target_col,
                         n_splits = n_splits,
                         missing_value = missing_value,
                         categorical_cols = categorical_cols,
                         feature_cols = feature_cols,
                         seed = seed)

Preparing data for k-fold cross-validation...
Training on 132473 samples with 9 features
Testing on 26495 samples
Features: ['PUFC05_AGE', 'PUFC06_MSTAT', 'PUFC04_SEX', 'PUFC07_GRADE', 'PUFC08_CURSCH', 'PUFC38_PREVJOB', 'PUFC31_FLWRK', 'PUFC30_LOOKW', 'PUFC34_WYNOT']
Training on 132473 samples with 9 features
Testing on 26495 samples
Features: ['PUFC05_AGE', 'PUFC06_MSTAT', 'PUFC04_SEX', 'PUFC07_GRADE', 'PUFC08_CURSCH', 'PUFC38_PREVJOB', 'PUFC31_FLWRK', 'PUFC30_LOOKW', 'PUFC34_WYNOT']
Training on 132473 samples with 9 features
Testing on 26495 samples
Features: ['PUFC05_AGE', 'PUFC06_MSTAT', 'PUFC04_SEX', 'PUFC07_GRADE', 'PUFC08_CURSCH', 'PUFC38_PREVJOB', 'PUFC31_FLWRK', 'PUFC30_LOOKW', 'PUFC34_WYNOT']
Training on 132473 samples with 9 features
Testing on 26495 samples
Features: ['PUFC05_AGE', 'PUFC06_MSTAT', 'PUFC04_SEX', 'PUFC07_GRADE', 'PUFC08_CURSCH', 'PUFC38_PREVJOB', 'PUFC31_FLWRK', 'PUFC30_LOOKW', 'PUFC34_WYNOT']
Training on 132474 samples with 9 features
Testing on 26494 sample

We will then make sure that Sample #6 is data the model will never see, thereby calling it exam while we train Samples #1 - #5

In [3]:
practice = folds_data[0 : 5]
exam = folds_data[5]

We will be redefining the train_model from trainEvalLR to fit our purposes, we will simply the train model to be used for testing. The only edits made for this block of code is at the end wherein we will return the model itself and the criterion used. Batch size and missing_value will be defined later. Moreover, we had kept the per epoch and accuracy display to show that the model is training properly.

In [15]:
from src.trainEvalLR import *

def practice_train_model(folds_data, learning_rate=0.01, batch_size=128, num_epochs=50,
                scheduler_step_size=5, scheduler_gamma=0.5, missing_value=-1,
                convergence_threshold=1e-4, patience=3, weight_decay=0,
                optimizer_string="sgd", seed=45):
    set_seeds(seed)
    fold_results = []
    all_y_test_aggregate = []
    all_predictions_aggregate = []
    
    all_final_train_losses = []
    all_final_test_losses = []
    all_final_train_accuracies = []
    all_final_test_accuracies = []

    #iterate over folds
    for fold_idx, fold in enumerate(folds_data, 1):
        print(f"\n{'='*20} FOLD {fold_idx} {'='*20}")
        train_loader, test_loader = prepare_data_loaders(fold['X_train'], fold['y_train'], fold['X_test'], fold['y_test'], batch_size, missing_value)
        input_dim = fold['X_train'].shape[1]
        model, criterion, optimizer, scheduler = initialize_model(
            input_dim, learning_rate, scheduler_step_size, scheduler_gamma, weight_decay=weight_decay,
            optimizer_name=optimizer_string, seed=seed
        )
        history = {'train_loss': [], 'test_loss': [], 'train_accuracy': [], 'test_accuracy': []}
        best_test_loss = float('inf')
        epochs_no_improve = 0

        #converge criteria
        for epoch in range(num_epochs):
            train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer)
            test_loss, test_accuracy, all_y_test, all_predictions = evaluate_model(model, test_loader, criterion)
            scheduler.step()
            history['train_loss'].append(train_loss)
            history['test_loss'].append(test_loss)
            history['train_accuracy'].append(train_accuracy)
            history['test_accuracy'].append(test_accuracy)
            print(f'Epoch {epoch+1}/{num_epochs}: Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}, '
                        f'Train Acc: {train_accuracy:.4f}, Test Acc: {test_accuracy:.4f}, LR: {scheduler.get_last_lr()[0]:.6f}')
            if test_loss < best_test_loss - convergence_threshold:
                best_test_loss = test_loss
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve == patience:
                    print(f'Early stopping triggered after {epoch+1} epochs.')
                    break
        
        #aggregate results
        all_final_train_losses.append(history['train_loss'][-1])
        all_final_test_losses.append(history['test_loss'][-1])
        all_final_train_accuracies.append(history['train_accuracy'][-1])
        all_final_test_accuracies.append(history['test_accuracy'][-1])
        
        #aggregate confusion matrix
        all_y_test_aggregate.extend(all_y_test)
        all_predictions_aggregate.extend(all_predictions)
        
        #log metrics
        cm = log_metrics(all_y_test, all_predictions)
        fold_results.append({'history': history, 'confusion_matrix': cm}) #append only once per fold.
    
    print(f"{'='*20} AGGREGATE RESULTS {'='*20}")
    aggregate_cm = log_metrics(all_y_test_aggregate, all_predictions_aggregate)
    
    #aggregate final metrics
    avg_test_accuracy = np.mean([res['history']['test_accuracy'][-1] for res in fold_results])
    avg_test_loss = np.mean([res['history']['test_loss'][-1] for res in fold_results])
    
    
    agg_final_train_loss = np.mean(all_final_train_losses)
    agg_final_test_loss = np.mean(all_final_test_losses)
    agg_final_train_accuracy = np.mean(all_final_train_accuracies)
    agg_final_test_accuracy = np.mean(all_final_test_accuracies)



    return {
        'avg_test_accuracy': avg_test_accuracy, 
        'avg_test_loss': avg_test_loss, 
        'fold_results': fold_results,
        'aggregate_confusion_matrix': aggregate_cm,
        'aggregated_final_metrics': {
            'avg_final_train_loss': agg_final_train_loss,
            'avg_final_test_loss': agg_final_test_loss,
            'avg_final_train_accuracy': agg_final_train_accuracy,
            'avg_final_test_accuracy': agg_final_test_accuracy
        },
        'all_final_test_accuracies' : all_final_test_accuracies, ## to be used for statistical tests,

        ## ONLY CHANGES MADE TO TRAIN_MODEL ARE THE RETURN VALUES BELOW
        'model' : model,
        'criterion' : criterion ## to be used for statistical tests
    }

After redefining our LR training model function, we can now train the model. We will only train with the practice set

In [17]:
result_dict = practice_train_model(
    practice,
    optimizer_string="sgd", #Stochastic Gradient Descent
    scheduler_step_size=5,
    learning_rate=0.01,
    scheduler_gamma=0.5,
    convergence_threshold=1e-4, 
    num_epochs=50,
    patience=3, # Stop at 3 epochs with no improvement
    weight_decay=0,
    seed=seed
)


==================== FOLD 1 ====================
Epoch 1/50: Train Loss: 0.1959, Test Loss: 0.1049, Train Acc: 0.9585, Test Acc: 0.9833, LR: 0.010000
Epoch 2/50: Train Loss: 0.0934, Test Loss: 0.0842, Train Acc: 0.9857, Test Acc: 0.9858, LR: 0.010000
Epoch 3/50: Train Loss: 0.0812, Test Loss: 0.0772, Train Acc: 0.9862, Test Acc: 0.9858, LR: 0.010000
Epoch 4/50: Train Loss: 0.0761, Test Loss: 0.0738, Train Acc: 0.9862, Test Acc: 0.9858, LR: 0.010000
Epoch 5/50: Train Loss: 0.0733, Test Loss: 0.0718, Train Acc: 0.9862, Test Acc: 0.9858, LR: 0.005000
Epoch 6/50: Train Loss: 0.0719, Test Loss: 0.0711, Train Acc: 0.9862, Test Acc: 0.9858, LR: 0.005000
Epoch 7/50: Train Loss: 0.0712, Test Loss: 0.0705, Train Acc: 0.9862, Test Acc: 0.9858, LR: 0.005000
Epoch 8/50: Train Loss: 0.0706, Test Loss: 0.0700, Train Acc: 0.9862, Test Acc: 0.9858, LR: 0.005000
Epoch 9/50: Train Loss: 0.0701, Test Loss: 0.0695, Train Acc: 0.9862, Test Acc: 0.9858, LR: 0.005000
Epoch 10/50: Train Loss: 0.0697, Test Los

We can now print out the test accuracy using our exam data

In [19]:
batch_size = 128
missing_value = -1
model = result_dict['model']
criterion = result_dict['criterion']

# Evaluate the model on the exam set
train_loader, test_loader = prepare_data_loaders(exam['X_train'], exam['y_train'], exam['X_test'], exam['y_test'], batch_size, missing_value)
test_loss, test_accuracy, all_y_test, all_predictions = evaluate_model(model, test_loader, criterion)
print('Test Accuracy with Exams: ', test_accuracy)
print('Test Loss with Exams: ', test_loss)

#conf matrix
cm = log_metrics(all_y_test, all_predictions)  

Test Accuracy with Exams:  0.9855061523363781
Test Loss with Exams:  0.07111682332962176

Confusion Matrix:
[[11809   186]
 [  198 14301]]

Classification Report:
              precision    recall  f1-score   support

         0.0  0.98350962 0.98449354 0.98400133     11995
         1.0  0.98716090 0.98634389 0.98675223     14499

    accuracy                      0.98550615     26494
   macro avg  0.98533526 0.98541871 0.98537678     26494
weighted avg  0.98550781 0.98550615 0.98550678     26494



This is enough to prove that the model performance, despite it experiencing data unseen to it provided that it is data completely from the samples, is good and does not experiencing data leakage. 